In [1]:
pip install kiteconnect pandas numpy yfinance scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.5/771.5 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 4.6 MB/s eta 0:00:00


In [2]:
from kiteconnect import KiteConnect

kite = KiteConnect(api_key="s9uc7uttlfq3i1ye")

# Step 3a: Generate login URL, open it in browser, get request_token
print(kite.login_url())



https://kite.zerodha.com/connect/login?api_key=s9uc7uttlfq3i1ye&v=3


In [3]:
# Step 3b: After login, paste the request_token here
request_token = "y1D2LV91sk8D7KVI266eCFhaw4RPeGl1"
data = kite.generate_session(request_token, api_secret="4jx7v45k1al5harirbiy4bk2fmqpb4wd")
kite.set_access_token(data["access_token"])

# Step 3c: Fetch holdings
holdings = kite.holdings()

import pandas as pd
df = pd.DataFrame(holdings)
print(df[["tradingsymbol", "quantity", "average_price", "last_price", "pnl"]])

   tradingsymbol  quantity  average_price  last_price           pnl
0           ADSL        10      79.755000      106.55    267.950000
1     BHARTIARTL        10    1904.100000     1871.45   -326.500000
2           BHEL         5      52.490000      258.95   1032.300000
3           BPCL         6     225.933333      352.75    760.900002
4        CEATLTD         1     952.750000     3457.80   2505.050000
5          CIPLA         4    1391.500000     1321.75   -279.000000
6          CUPID        25       0.000000      402.20  10055.000000
7        DRREDDY         4    1166.900000     1304.00    548.400000
8     EQUITASBNK        46      46.834050       59.79    595.973700
9        ETERNAL         8      72.681250      232.57   1279.110000
10    GREAVESCOT        12     152.504167      154.08     18.909996
11          HFCL        12      77.066667       69.75    -87.800004
12          IDEA        23       8.473478       10.06     36.490006
13          IFCI         8      50.877500       

In [11]:
# ── 3. DOWNLOAD NIFTY 50 BENCHMARK ────────────────────────────────────
print("\n📥 Downloading Nifty 50 data...")
nifty_raw = yf.download("^NSEI", period="1y", interval="1d",
                         auto_adjust=True, progress=False)

nifty_close = nifty_raw["Close"]
if isinstance(nifty_close, pd.DataFrame):
    nifty_close = nifty_close.iloc[:, 0]
nifty_close.index = pd.to_datetime(nifty_close.index)
if isinstance(nifty_close.index, pd.MultiIndex):
    nifty_close.index = nifty_close.index.get_level_values("Date")

nifty_returns = nifty_close.pct_change().dropna()
print(f"✅ Nifty 50: {len(nifty_returns)} rows")


📥 Downloading Nifty 50 data...
✅ Nifty 50: 246 rows


In [31]:
# ── 4. DOWNLOAD STOCK DATA ─────────────────────────────────────────────
print("\n📥 Downloading stock data...")
stock_returns_dict = {}
stock_weights      = {}
failed_stocks      = []
EXCLUDE = ["BHARTIARTL"]

for _, row in df.iterrows():
    symbol   = row["tradingsymbol"]
    yf_symbol = symbol + ".NS"
    print(yf_symbol)
    value    = row["quantity"] * row["last_price"]

    if symbol in EXCLUDE:
        print(f"⏭️  Skipping {symbol} — manually excluded")
        continue

    try:
        stock_data = yf.download(
            yf_symbol,
            period="1y",
            interval="1d",
            auto_adjust=True,
            progress=False,
            #group_by="ticker
        )

        # Skip if not enough history
        if stock_data.empty or len(stock_data) < 200:
            print(f"⚠️  Skipping {symbol} — not enough data ({len(stock_data)} rows)")
            failed_stocks.append(symbol)
            continue

        # Flatten MultiIndex → plain Series
        close = stock_data["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close.index = pd.to_datetime(close.index)
        if isinstance(close.index, pd.MultiIndex):
            close.index = close.index.get_level_values("Date")

        returns = close.pct_change().dropna()

        if returns.empty:
            print(f"⚠️  Skipping {symbol} — empty returns after pct_change")
            failed_stocks.append(symbol)
            continue

        stock_returns_dict[symbol] = returns
        stock_weights[symbol]      = value
        print(f"✅ {symbol}: {len(returns)} rows | ₹{value:,.0f} weight")

    except Exception as e:
        print(f"❌ Error fetching {symbol}: {e}")
        failed_stocks.append(symbol)

print(f"\n✅ Loaded: {len(stock_returns_dict)} stocks")
print(f"⚠️  Skipped: {failed_stocks}")

if not stock_returns_dict:
    raise RuntimeError("No stock data loaded. Check your symbols or internet connection.")




📥 Downloading stock data...
ADSL.NS
✅ ADSL: 247 rows | ₹1,066 weight
BHARTIARTL.NS
⏭️  Skipping BHARTIARTL — manually excluded
BHEL.NS
✅ BHEL: 247 rows | ₹1,295 weight
BPCL.NS
✅ BPCL: 247 rows | ₹2,116 weight
CEATLTD.NS
✅ CEATLTD: 247 rows | ₹3,458 weight
CIPLA.NS
✅ CIPLA: 247 rows | ₹5,287 weight
CUPID.NS
✅ CUPID: 247 rows | ₹10,055 weight
DRREDDY.NS
✅ DRREDDY: 247 rows | ₹5,216 weight
EQUITASBNK.NS
✅ EQUITASBNK: 247 rows | ₹2,750 weight
ETERNAL.NS
✅ ETERNAL: 246 rows | ₹1,861 weight
GREAVESCOT.NS
✅ GREAVESCOT: 247 rows | ₹1,849 weight
HFCL.NS
✅ HFCL: 247 rows | ₹837 weight
IDEA.NS
✅ IDEA: 247 rows | ₹231 weight
IFCI.NS
✅ IFCI: 247 rows | ₹437 weight
IOB.NS
✅ IOB: 247 rows | ₹3,408 weight
IOC.NS
✅ IOC: 247 rows | ₹1,181 weight
IRFC.NS
✅ IRFC: 247 rows | ₹1,094 weight
ITC.NS
✅ ITC: 247 rows | ₹3,098 weight
ITCHOTELS.NS
✅ ITCHOTELS: 245 rows | ₹167 weight
JSWENERGY.NS
✅ JSWENERGY: 247 rows | ₹2,437 weight
KPITTECH.NS
✅ KPITTECH: 247 rows | ₹3,494 weight
MANINDS.NS
✅ MANINDS: 247 rows |

In [32]:
# ── 5. BUILD ALIGNED RETURNS DATAFRAME ────────────────────────────────
returns_df = pd.DataFrame(stock_returns_dict).dropna()
print(f"\n📊 Aligned DataFrame: {returns_df.shape}")
print(f"📅 Date range: {returns_df.index[0].date()} to {returns_df.index[-1].date()}")




📊 Aligned DataFrame: (245, 38)
📅 Date range: 2025-03-07 to 2026-03-06


In [33]:
# ── 6. CALCULATE WEIGHTED PORTFOLIO RETURNS ───────────────────────────
total_value = sum(stock_weights[s] for s in returns_df.columns)
weights     = np.array([stock_weights[s] / total_value for s in returns_df.columns])

portfolio_daily_returns = returns_df.dot(weights)



In [34]:
# ── 7. ALIGN WITH NIFTY & CALCULATE BETA ─────────────────────────────
combined = pd.concat([portfolio_daily_returns, nifty_returns], axis=1).dropna()
combined.columns = ["portfolio", "nifty"]

print(f"📅 Common trading days with Nifty: {len(combined)}")

cov_matrix = np.cov(combined["portfolio"], combined["nifty"])
beta       = cov_matrix[0, 1] / cov_matrix[1, 1]



📅 Common trading days with Nifty: 245


In [35]:
# ── 8. CALCULATE ALPHA ─────────────────────────────────────────────────
risk_free_rate          = 0.0668                                   # ~6.5% Indian T-bill
portfolio_annual_return = combined["portfolio"].mean() * 252
market_annual_return    = combined["nifty"].mean()    * 252
alpha = portfolio_annual_return - (risk_free_rate + beta * (market_annual_return - risk_free_rate))



In [36]:
# ── 9. PRINT FINAL RESULTS ─────────────────────────────────────────────
print("\n" + "="*45)
print("        📊 PORTFOLIO ANALYSIS RESULTS")
print("="*45)
print(f"  Stocks analysed   : {len(returns_df.columns)}")
print(f"  Stocks skipped    : {len(failed_stocks)} {failed_stocks}")
print(f"  Portfolio value   : ₹{total_value:,.0f}")
print("-"*45)
print(f"  Beta              : {beta:.2f}")
print(f"  Portfolio return  : {portfolio_annual_return*100:.2f}% p.a.")
print(f"  Nifty return      : {market_annual_return*100:.2f}% p.a.")
print(f"  Alpha             : {alpha*100:.2f}% p.a.")
print("="*45)

if alpha > 0:
    print(f"\n🎉 Your portfolio is BEATING the market by {alpha*100:.2f}%!")
elif alpha < 0:
    print(f"\n📉 Your portfolio is UNDERPERFORMING the market by {abs(alpha)*100:.2f}%")
else:
    print(f"\n➡️  Your portfolio is matching the market")

if beta > 1.2:
    print(f"⚡ High beta ({beta:.2f}) — your portfolio is more volatile than Nifty")
elif beta < 0.8:
    print(f"🛡️  Low beta ({beta:.2f}) — your portfolio is more stable than Nifty")
else:
    print(f"⚖️  Moderate beta ({beta:.2f}) — moves roughly in line with Nifty")


        📊 PORTFOLIO ANALYSIS RESULTS
  Stocks analysed   : 38
  Stocks skipped    : 1 ['TATACAP']
  Portfolio value   : ₹79,200
---------------------------------------------
  Beta              : 1.11
  Portfolio return  : 19.26% p.a.
  Nifty return      : 9.04% p.a.
  Alpha             : 9.97% p.a.

🎉 Your portfolio is BEATING the market by 9.97%!
⚖️  Moderate beta (1.11) — moves roughly in line with Nifty
